In [ ]:
# =============================================================================
# CÉLULA 1 — SETUP & CONFIGURAÇÃO DA API
#
# Instala dependências, configura o acesso ao Gemma via Gemini API,
# e define o modo de fallback deterministico para execução offline.
# =============================================================================

import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install("google-generativeai")

import os
import re
import json
import math
import time
import hashlib
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict

import google.generativeai as genai

# ── Configuração da API (Kaggle Secrets ou variável de ambiente) ──────────────
GEMINI_API_KEY = None

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    GEMINI_API_KEY = secrets.get_secret("GEMINI_API_KEY")
    print("[OK] GEMINI_API_KEY carregada via Kaggle Secrets")
except Exception:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
    if GEMINI_API_KEY:
        print("[OK] GEMINI_API_KEY carregada via variável de ambiente")
    else:
        print("[AVISO] GEMINI_API_KEY não encontrada — modo FALLBACK ativado.")
        print("        O notebook executa do início ao fim com vetores deterministicos.")
        print("        Para usar o Gemma real: configure GEMINI_API_KEY em Kaggle Secrets.")

GEMMA_MODE = "api" if GEMINI_API_KEY else "fallback"

if GEMMA_MODE == "api":
    genai.configure(api_key=GEMINI_API_KEY)
    gemma_model = genai.GenerativeModel("gemini-2.0-flash-lite")
    print(f"[OK] Modo: Gemma via Gemini API (gemini-2.0-flash-lite)")
else:
    gemma_model = None
    print(f"[OK] Modo: fallback deterministico por hash")

print(f"\n[PRONTO] Todas as dependências carregadas.")

In [ ]:
# =============================================================================
# CÉLULA 2 — ARQUITETURA DO ALGORITMO JP: DUAS FASES
#
# Explicitamos aqui a premissa central do trabalho e por que a abordagem
# com TDA + Wasserstein difere fundamentalmente de sistemas de embedding
# convencionais.
# =============================================================================

print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║          ALGORITMO JP — INTELIGÊNCIA ARTIFICIAL PICTÓRICA (IAP)             ║
║          João Pedro Pereira Passos · UFT · Hackathon Gemma 4 Good 2026      ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  PROBLEMA: LLMs convencionais operam com bilhões de parâmetros a CADA       ║
║  inferência — custo enorme, distribuído, opaco. Para CAA (Comunicação       ║
║  Aumentativa e Alternativa), o sistema precisa ser responsivo, leve e       ║
║  matematicamente verificável.                                                ║
║                                                                              ║
║  SOLUÇÃO: O Algoritmo JP propõe uma arquitetura em duas fases separadas:    ║
║                                                                              ║
║  ┌─────────────────────────────────────────────────────────────────────┐    ║
║  │ FASE 1 — PRÉ-CAMADA SEMÂNTICA (Gemma, executa UMA VEZ, offline)    │    ║
║  │                                                                     │    ║
║  │  ícone ("comer") + contexto IAP                                     │    ║
║  │         ↓                                                           │    ║
║  │    Gemma analisa label, categoria, relações semânticas              │    ║
║  │         ↓                                                           │    ║
║  │    Vetor 12D semântico real (scores 1-10 por dimensão IAP)         │    ║
║  │    Ex: {alimentacao: 9.2, acao: 7.8, corpo: 4.1, ...}             │    ║
║  │         ↓                                                           │    ║
║  │    Resultado armazenado (cache permanente)                          │    ║
║  └─────────────────────────────────────────────────────────────────────┘    ║
║                                                                              ║
║  ┌─────────────────────────────────────────────────────────────────────┐    ║
║  │ FASE 2 — INTELIGÊNCIA FLUIDA (matemática exata, runtime)           │    ║
║  │                                                                     │    ║
║  │  Vetores 12D (pré-calculados)                                      │    ║
║  │         ↓                                                           │    ║
║  │    Diagramas de Persistência H0+H1 (homologia persistente)         │    ║
║  │         ↓                                                           │    ║
║  │    Distância de Wasserstein entre diagramas (O(n²), exata)        │    ║
║  │         ↓                                                           │    ║
║  │    MDS clássico → coordenadas 2D                                   │    ║
║  │         ↓                                                           │    ║
║  │    Atlas IAP: relevo do conhecimento navegável                      │    ║
║  └─────────────────────────────────────────────────────────────────────┘    ║
║                                                                              ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  POR QUE ESTA ABORDAGEM É ORIGINAL?                                         ║
║                                                                              ║
║  A separação offline/online em si NÃO é nova (RAG, SBERT fazem isso).      ║
║  A novidade está em O QUE ACONTECE ENTRE o LLM e o runtime:                ║
║                                                                              ║
║  Sistemas convencionais:                                                     ║
║    LLM → vetor → similaridade por COSSENO (mede DIREÇÃO)                   ║
║                                                                              ║
║  Algoritmo JP:                                                               ║
║    LLM → vetor → DIAGRAMA DE PERSISTÊNCIA → WASSERSTEIN (mede FORMA)       ║
║                                                                              ║
║  A persistência homológica captura ESTRUTURAS RELACIONAIS que o cosseno     ║
║  perde: dois ícones podem ter vetores na mesma direção mas "formas          ║
║  semânticas" radicalmente diferentes.                                        ║
║                                                                              ║
║  Aplicado a CAA/pictogramas: inédito na literatura (2026).                  ║
╚══════════════════════════════════════════════════════════════════════════════╝
""")

In [ ]:
# =============================================================================
# CÉLULA 3 — 12 DIMENSÕES IAP + VETORES ESTÁTICOS DE REFERÊNCIA
#
# Define as 12 dimensões semânticas com descrições ricas usadas no
# prompt ao Gemma. Contrasta com os vetores hardcoded (referência).
# =============================================================================

# Dimensões IAP com descrições para o prompt ao Gemma
IAP_DIMENSOES = {
    "comunicacao":     "comunicação verbal/não-verbal, fala, linguagem, gestos, AAC",
    "emocao":          "emoções, sentimentos, expressões afetivas, estados psicológicos",
    "corpo":           "partes do corpo, anatomia, funções corporais, sensações físicas",
    "alimentacao":     "comida, bebida, nutrição, refeições, fome, saciedade",
    "familia_pessoas": "pessoas, relações interpessoais, família, papéis sociais",
    "acao":            "ações, verbos, movimentos, processos dinâmicos, atividades",
    "lugar":           "espaços, ambientes, locais, posição, contexto espacial",
    "saude":           "saúde, medicina, terapia, cuidado, higiene, bem-estar físico",
    "natureza":        "natureza, animais, clima, ambiente natural, ecologia",
    "tempo":           "tempo, datas, sequências, duração, rotinas, calendário",
    "objeto":          "objetos físicos concretos, ferramentas, itens cotidianos",
    "escola":          "educação, aprendizado, estudo, escola, conhecimento formal",
}

DIM_KEYS = list(IAP_DIMENSOES.keys())
DIM = len(DIM_KEYS)  # 12

# Vetores ESTÁTICOS hardcoded (referência para comparação — abordagem ATUAL)
# Regra fixa: 9=categoria própria, 2=vizinhos, 1=demais
STATIC_VECTORS = {
    "comunicacao":     [9, 2, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1],
    "emocao":          [2, 9, 2, 1, 1, 1, 2, 1, 1, 1, 1, 1],
    "corpo":           [1, 2, 9, 2, 1, 1, 1, 2, 1, 1, 1, 1],
    "alimentacao":     [1, 1, 2, 9, 2, 1, 1, 1, 2, 1, 1, 1],
    "familia_pessoas": [1, 1, 1, 2, 9, 2, 1, 1, 1, 2, 1, 1],
    "acao":            [1, 1, 1, 1, 2, 9, 2, 1, 1, 1, 2, 1],
    "lugar":           [1, 1, 1, 1, 1, 2, 9, 2, 1, 1, 1, 2],
    "saude":           [1, 1, 1, 1, 1, 1, 2, 9, 2, 1, 1, 1],
    "natureza":        [1, 1, 1, 1, 1, 1, 1, 2, 9, 2, 1, 1],
    "tempo":           [1, 1, 1, 1, 1, 1, 1, 1, 2, 9, 2, 1],
    "objeto":          [1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 9, 2],
    "escola":          [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 9],
}

def static_vec(categoria, icon_id):
    """Vetor estático com ruído deterministico por ID (abordagem anterior)."""
    base = STATIC_VECTORS.get(categoria, [5] * DIM)
    nid = int(re.sub(r'\D', '', str(icon_id)) or '0') % 10000
    noise = (nid % 7) / 10
    return [max(1.0, v + noise * math.sin(i * (nid % 13))) for i, v in enumerate(base)]

# Exibe a tabela de dimensões
print("=" * 72)
print("  12 DIMENSÕES SEMÂNTICAS IAP — Usadas no prompt ao Gemma")
print("=" * 72)
for i, (key, desc) in enumerate(IAP_DIMENSOES.items(), 1):
    print(f"  {i:2d}. {key:<18} → {desc}")
print()
print("  CONTRASTE — vetor ESTÁTICO para 'comer' (cat: alimentacao):")
sv = static_vec("alimentacao", "999")
print("  " + " ".join(f"{k[:3]}:{v:.1f}" for k, v in zip(DIM_KEYS, sv)))
print()
print("  ESPERADO — vetor GEMMA para 'comer':")
print("  alimentacao alto (~9), acao alto (~8), corpo moderado (~4)")
print("  → impossível com regras fixas baseadas apenas na categoria")

In [ ]:
# =============================================================================
# CÉLULA 4 — CARREGAMENTO DE ÍCONES (Noun Project + ARASAAC)
#
# Carrega ~130 ícones de duas fontes: Noun Project (10/categoria × 12 cats)
# e ARASAAC (27 ícones clássicos de CAA). Dataset embarcado para portabilidade.
# =============================================================================

# Dataset embarcado — gerado a partir de noun_atlas_data.json + atlas_data.json
# (disponíveis no repositório GitHub: joaopedropassostocantins/AFASIA, branch disfasia)
ICONS_RAW = [
    {
        "id": "8195788",
        "palavra": "speech",
        "categoria": "comunicacao",
        "fonte": "noun"
    },
    {
        "id": "8333072",
        "palavra": "talking",
        "categoria": "comunicacao",
        "fonte": "noun"
    },
    {
        "id": "7544702",
        "palavra": "talk",
        "categoria": "comunicacao",
        "fonte": "noun"
    },
    {
        "id": "3998230",
        "palavra": "sign language",
        "categoria": "comunicacao",
        "fonte": "noun"
    },
    {
        "id": "6749508",
        "palavra": "gestures",
        "categoria": "comunicacao",
        "fonte": "noun"
    },
    {
        "id": "6749416",
        "palavra": "gesture",
        "categoria": "comunicacao",
        "fonte": "noun"
    },
    {
        "id": "5776693",
        "palavra": "gesturing",
        "categoria": "comunicacao",
        "fonte": "noun"
    },
    {
        "id": "8235493",
        "palavra": "phone call",
        "categoria": "emocao",
        "fonte": "noun"
    },
    {
        "id": "8205041",
        "palavra": "angry",
        "categoria": "emocao",
        "fonte": "noun"
    },
    {
        "id": "7981946",
        "palavra": "emotion",
        "categoria": "emocao",
        "fonte": "noun"
    },
    {
        "id": "6480295",
        "palavra": "emotions",
        "categoria": "emocao",
        "fonte": "noun"
    },
    {
        "id": "8233120",
        "palavra": "sad",
        "categoria": "emocao",
        "fonte": "noun"
    },
    {
        "id": "8357919",
        "palavra": "sadness",
        "categoria": "emocao",
        "fonte": "noun"
    },
    {
        "id": "8067602",
        "palavra": "happy",
        "categoria": "emocao",
        "fonte": "noun"
    },
    {
        "id": "8338287",
        "palavra": "feeling",
        "categoria": "emocao",
        "fonte": "noun"
    },
    {
        "id": "2952744",
        "palavra": "body",
        "categoria": "corpo",
        "fonte": "noun"
    },
    {
        "id": "8326685",
        "palavra": "hand",
        "categoria": "corpo",
        "fonte": "noun"
    },
    {
        "id": "8313120",
        "palavra": "hands",
        "categoria": "corpo",
        "fonte": "noun"
    },
    {
        "id": "8102272",
        "palavra": "ear",
        "categoria": "corpo",
        "fonte": "noun"
    },
    {
        "id": "8076024",
        "palavra": "earings",
        "categoria": "corpo",
        "fonte": "noun"
    },
    {
        "id": "7916738",
        "palavra": "earing",
        "categoria": "corpo",
        "fonte": "noun"
    },
    {
        "id": "7914185",
        "palavra": "ears",
        "categoria": "corpo",
        "fonte": "noun"
    },
    {
        "id": "8267105",
        "palavra": "eye",
        "categoria": "corpo",
        "fonte": "noun"
    },
    {
        "id": "8133979",
        "palavra": "face",
        "categoria": "corpo",
        "fonte": "noun"
    },
    {
        "id": "8101317",
        "palavra": "faces",
        "categoria": "corpo",
        "fonte": "noun"
    },
    {
        "id": "8228221",
        "palavra": "food",
        "categoria": "alimentacao",
        "fonte": "noun"
    },
    {
        "id": "1235535",
        "palavra": "water water",
        "categoria": "alimentacao",
        "fonte": "noun"
    },
    {
        "id": "8287001",
        "palavra": "water",
        "categoria": "alimentacao",
        "fonte": "noun"
    },
    {
        "id": "5079785",
        "palavra": "watering",
        "categoria": "alimentacao",
        "fonte": "noun"
    },
    {
        "id": "8265395",
        "palavra": "drink",
        "categoria": "alimentacao",
        "fonte": "noun"
    },
    {
        "id": "8260253",
        "palavra": "drinking",
        "categoria": "alimentacao",
        "fonte": "noun"
    },
    {
        "id": "8239728",
        "palavra": "drinks",
        "categoria": "alimentacao",
        "fonte": "noun"
    },
    {
        "id": "8357536",
        "palavra": "fruit",
        "categoria": "alimentacao",
        "fonte": "noun"
    },
    {
        "id": "5208882",
        "palavra": "bread",
        "categoria": "alimentacao",
        "fonte": "noun"
    },
    {
        "id": "6187604",
        "palavra": "mother",
        "categoria": "familia_pessoas",
        "fonte": "noun"
    },
    {
        "id": "8334435",
        "palavra": "baby",
        "categoria": "familia_pessoas",
        "fonte": "noun"
    },
    {
        "id": "5218380",
        "palavra": "child",
        "categoria": "familia_pessoas",
        "fonte": "noun"
    },
    {
        "id": "5295967",
        "palavra": "father",
        "categoria": "familia_pessoas",
        "fonte": "noun"
    },
    {
        "id": "6722117",
        "palavra": "mother's",
        "categoria": "familia_pessoas",
        "fonte": "noun"
    },
    {
        "id": "8214117",
        "palavra": "family",
        "categoria": "familia_pessoas",
        "fonte": "noun"
    },
    {
        "id": "6815413",
        "palavra": "people",
        "categoria": "acao",
        "fonte": "noun"
    },
    {
        "id": "7437871",
        "palavra": "run",
        "categoria": "acao",
        "fonte": "noun"
    },
    {
        "id": "8226678",
        "palavra": "sit",
        "categoria": "acao",
        "fonte": "noun"
    },
    {
        "id": "2812085",
        "palavra": "sit-in",
        "categoria": "acao",
        "fonte": "noun"
    },
    {
        "id": "6723653",
        "palavra": "walk",
        "categoria": "acao",
        "fonte": "noun"
    },
    {
        "id": "6202566",
        "palavra": "jumping",
        "categoria": "acao",
        "fonte": "noun"
    },
    {
        "id": "6070087",
        "palavra": "jump",
        "categoria": "acao",
        "fonte": "noun"
    },
    {
        "id": "8141574",
        "palavra": "sleep",
        "categoria": "acao",
        "fonte": "noun"
    },
    {
        "id": "8096367",
        "palavra": "sleeping",
        "categoria": "acao",
        "fonte": "noun"
    },
    {
        "id": "8337042",
        "palavra": "hospital",
        "categoria": "lugar",
        "fonte": "noun"
    },
    {
        "id": "6807924",
        "palavra": "street",
        "categoria": "lugar",
        "fonte": "noun"
    },
    {
        "id": "6280301",
        "palavra": "streets",
        "categoria": "lugar",
        "fonte": "noun"
    },
    {
        "id": "8185565",
        "palavra": "park",
        "categoria": "lugar",
        "fonte": "noun"
    },
    {
        "id": "8321720",
        "palavra": "school",
        "categoria": "lugar",
        "fonte": "noun"
    },
    {
        "id": "8325020",
        "palavra": "home",
        "categoria": "lugar",
        "fonte": "noun"
    },
    {
        "id": "8172294",
        "palavra": "sick",
        "categoria": "saude",
        "fonte": "noun"
    },
    {
        "id": "7030191",
        "palavra": "pain",
        "categoria": "saude",
        "fonte": "noun"
    },
    {
        "id": "8036395",
        "palavra": "doctors",
        "categoria": "saude",
        "fonte": "noun"
    },
    {
        "id": "7569284",
        "palavra": "doctor",
        "categoria": "saude",
        "fonte": "noun"
    },
    {
        "id": "8220409",
        "palavra": "health",
        "categoria": "saude",
        "fonte": "noun"
    },
    {
        "id": "7933744",
        "palavra": "medicine",
        "categoria": "saude",
        "fonte": "noun"
    },
    {
        "id": "8295397",
        "palavra": "medicines",
        "categoria": "saude",
        "fonte": "noun"
    },
    {
        "id": "7834260",
        "palavra": "pill",
        "categoria": "saude",
        "fonte": "noun"
    },
    {
        "id": "7660792",
        "palavra": "pills",
        "categoria": "saude",
        "fonte": "noun"
    },
    {
        "id": "8058319",
        "palavra": "nurse",
        "categoria": "saude",
        "fonte": "noun"
    },
    {
        "id": "8196764",
        "palavra": "call phone",
        "categoria": "natureza",
        "fonte": "noun"
    },
    {
        "id": "8219466",
        "palavra": "rain",
        "categoria": "natureza",
        "fonte": "noun"
    },
    {
        "id": "8188113",
        "palavra": "raining",
        "categoria": "natureza",
        "fonte": "noun"
    },
    {
        "id": "8355756",
        "palavra": "flower",
        "categoria": "natureza",
        "fonte": "noun"
    },
    {
        "id": "5067086",
        "palavra": "flowers",
        "categoria": "natureza",
        "fonte": "noun"
    },
    {
        "id": "4988973",
        "palavra": "bluebell",
        "categoria": "natureza",
        "fonte": "noun"
    },
    {
        "id": "8299740",
        "palavra": "sun",
        "categoria": "natureza",
        "fonte": "noun"
    },
    {
        "id": "8250664",
        "palavra": "tree",
        "categoria": "natureza",
        "fonte": "noun"
    },
    {
        "id": "8340832",
        "palavra": "trees",
        "categoria": "natureza",
        "fonte": "noun"
    },
    {
        "id": "8346241",
        "palavra": "nature",
        "categoria": "natureza",
        "fonte": "noun"
    },
    {
        "id": "8259412",
        "palavra": "morning",
        "categoria": "tempo",
        "fonte": "noun"
    },
    {
        "id": "3202622",
        "palavra": "day",
        "categoria": "tempo",
        "fonte": "noun"
    },
    {
        "id": "5864470",
        "palavra": "not the day",
        "categoria": "tempo",
        "fonte": "noun"
    },
    {
        "id": "8344255",
        "palavra": "time",
        "categoria": "tempo",
        "fonte": "noun"
    },
    {
        "id": "8333977",
        "palavra": "on time",
        "categoria": "tempo",
        "fonte": "noun"
    },
    {
        "id": "8076120",
        "palavra": "time and time",
        "categoria": "tempo",
        "fonte": "noun"
    },
    {
        "id": "8333411",
        "palavra": "clock",
        "categoria": "tempo",
        "fonte": "noun"
    },
    {
        "id": "8047748",
        "palavra": "night",
        "categoria": "tempo",
        "fonte": "noun"
    },
    {
        "id": "2143040",
        "palavra": "tomorrow",
        "categoria": "tempo",
        "fonte": "noun"
    },
    {
        "id": "7250258",
        "palavra": "tomorrow list",
        "categoria": "tempo",
        "fonte": "noun"
    },
    {
        "id": "8082938",
        "palavra": "phone",
        "categoria": "objeto",
        "fonte": "noun"
    },
    {
        "id": "8327128",
        "palavra": "table",
        "categoria": "objeto",
        "fonte": "noun"
    },
    {
        "id": "1299598",
        "palavra": "pen",
        "categoria": "objeto",
        "fonte": "noun"
    },
    {
        "id": "8355644",
        "palavra": "book",
        "categoria": "objeto",
        "fonte": "noun"
    },
    {
        "id": "8268403",
        "palavra": "computer",
        "categoria": "objeto",
        "fonte": "noun"
    },
    {
        "id": "7990153",
        "palavra": "toys",
        "categoria": "objeto",
        "fonte": "noun"
    },
    {
        "id": "7787910",
        "palavra": "toy",
        "categoria": "objeto",
        "fonte": "noun"
    },
    {
        "id": "8285872",
        "palavra": "key",
        "categoria": "objeto",
        "fonte": "noun"
    },
    {
        "id": "7954433",
        "palavra": "clothes",
        "categoria": "objeto",
        "fonte": "noun"
    },
    {
        "id": "1392546",
        "palavra": "bag",
        "categoria": "objeto",
        "fonte": "noun"
    },
    {
        "id": "7070515",
        "palavra": "class",
        "categoria": "escola",
        "fonte": "noun"
    },
    {
        "id": "8104917",
        "palavra": "classes",
        "categoria": "escola",
        "fonte": "noun"
    },
    {
        "id": "8340960",
        "palavra": "student",
        "categoria": "escola",
        "fonte": "noun"
    },
    {
        "id": "1111422",
        "palavra": "teacher",
        "categoria": "escola",
        "fonte": "noun"
    },
    {
        "id": "7489630",
        "palavra": "study",
        "categoria": "escola",
        "fonte": "noun"
    },
    {
        "id": "8140966",
        "palavra": "students",
        "categoria": "escola",
        "fonte": "noun"
    },
    {
        "id": "7924211",
        "palavra": "number",
        "categoria": "escola",
        "fonte": "noun"
    },
    {
        "id": "6440950",
        "palavra": "numbered",
        "categoria": "escola",
        "fonte": "noun"
    },
    {
        "id": "3808808",
        "palavra": "numbering",
        "categoria": "escola",
        "fonte": "noun"
    },
    {
        "id": "4485956",
        "palavra": "number one",
        "categoria": "escola",
        "fonte": "noun"
    },
    {
        "id": "ar32464",
        "palavra": "agua",
        "categoria": "alimentacao",
        "fonte": "arasaac"
    },
    {
        "id": "ar4611",
        "palavra": "comida",
        "categoria": "alimentacao",
        "fonte": "arasaac"
    },
    {
        "id": "ar35559",
        "palavra": "fome",
        "categoria": "alimentacao",
        "fonte": "arasaac"
    },
    {
        "id": "ar2700",
        "palavra": "maca",
        "categoria": "alimentacao",
        "fonte": "arasaac"
    },
    {
        "id": "ar10741",
        "palavra": "remedio",
        "categoria": "alimentacao",
        "fonte": "arasaac"
    },
    {
        "id": "ar10840",
        "palavra": "banheiro",
        "categoria": "alimentacao",
        "fonte": "arasaac"
    },
    {
        "id": "ar12252",
        "palavra": "ajuda",
        "categoria": "alimentacao",
        "fonte": "arasaac"
    },
    {
        "id": "ar6323",
        "palavra": "dormir",
        "categoria": "alimentacao",
        "fonte": "arasaac"
    },
    {
        "id": "ar9907",
        "palavra": "feliz",
        "categoria": "emocao",
        "fonte": "arasaac"
    },
    {
        "id": "ar35545",
        "palavra": "triste",
        "categoria": "emocao",
        "fonte": "arasaac"
    },
    {
        "id": "ar2367",
        "palavra": "dor",
        "categoria": "emocao",
        "fonte": "arasaac"
    },
    {
        "id": "ar10261",
        "palavra": "medo",
        "categoria": "emocao",
        "fonte": "arasaac"
    },
    {
        "id": "ar35537",
        "palavra": "cansado",
        "categoria": "emocao",
        "fonte": "arasaac"
    },
    {
        "id": "ar6964",
        "palavra": "casa",
        "categoria": "lugar",
        "fonte": "arasaac"
    },
    {
        "id": "ar36210",
        "palavra": "hospital",
        "categoria": "lugar",
        "fonte": "arasaac"
    },
    {
        "id": "ar32446",
        "palavra": "escola",
        "categoria": "lugar",
        "fonte": "arasaac"
    },
    {
        "id": "ar6561",
        "palavra": "medico",
        "categoria": "familia_pessoas",
        "fonte": "arasaac"
    },
    {
        "id": "ar38351",
        "palavra": "familia",
        "categoria": "familia_pessoas",
        "fonte": "arasaac"
    },
    {
        "id": "ar25790",
        "palavra": "amigo",
        "categoria": "familia_pessoas",
        "fonte": "arasaac"
    },
    {
        "id": "ar2458",
        "palavra": "mae",
        "categoria": "familia_pessoas",
        "fonte": "arasaac"
    },
    {
        "id": "ar2497",
        "palavra": "pai",
        "categoria": "familia_pessoas",
        "fonte": "arasaac"
    },
    {
        "id": "ar5441",
        "palavra": "quero",
        "categoria": "acao",
        "fonte": "arasaac"
    },
    {
        "id": "ar5584",
        "palavra": "sim",
        "categoria": "acao",
        "fonte": "arasaac"
    },
    {
        "id": "ar5526",
        "palavra": "nao",
        "categoria": "acao",
        "fonte": "arasaac"
    },
    {
        "id": "ar7196",
        "palavra": "parar",
        "categoria": "acao",
        "fonte": "arasaac"
    },
    {
        "id": "ar8142",
        "palavra": "ir",
        "categoria": "acao",
        "fonte": "arasaac"
    },
    {
        "id": "ar6456",
        "palavra": "comer",
        "categoria": "acao",
        "fonte": "arasaac"
    }
]

# Normaliza: remove duplicatas de palavra por categoria
seen_palavras = set()
ICONS = []
for icon in ICONS_RAW:
    key = (icon["palavra"].lower().strip(), icon["categoria"])
    if key not in seen_palavras:
        seen_palavras.add(key)
        ICONS.append(icon)

print(f"Total de ícones carregados: {len(ICONS)}")
print()

# Distribuição por categoria e fonte
from collections import Counter
cat_counts = Counter(i["categoria"] for i in ICONS)
fonte_counts = Counter(i["fonte"] for i in ICONS)

print("Distribuição por categoria:")
for cat, cnt in sorted(cat_counts.items()):
    bar = "█" * cnt
    print(f"  {cat:<20} {cnt:3d}  {bar}")

print()
print(f"Fontes: Noun Project={fonte_counts.get('noun',0)} | ARASAAC={fonte_counts.get('arasaac',0)}")

# Ícones cross-categoria de interesse (demonstrarão melhor o Gemma)
CROSS_CAT_FOCUS = ["comer", "medico", "water", "sleep", "family", "run", "hospital", "doctor", "ajuda"]
print()
print("Ícones cross-categoria selecionados para análise:")
for icon in ICONS:
    if any(w in icon["palavra"].lower() for w in CROSS_CAT_FOCUS):
        print(f"  {icon['palavra']:<20} [{icon['categoria']}] ({icon['fonte']})")

In [ ]:
# =============================================================================
# CÉLULA 5 — PRÉ-CAMADA GEMMA: VETORES SEMÂNTICOS DINÂMICOS
#
# FASE 1 DO ALGORITMO JP — O Gemma analisa cada ícone e retorna um vetor
# 12D com scores reais. Cache evita re-chamadas. Fallback deterministico
# garante execução mesmo sem API key.
# =============================================================================

import time

# ── Prompt template ──────────────────────────────────────────────────────────
PROMPT_TEMPLATE = """Você é um analisador semântico especializado em pictogramas de comunicação aumentativa e alternativa (CAA) para pessoas com afasia e disfasia.

Para o ícone "{palavra}" (categoria principal: {categoria}), forneça uma pontuação de 1.0 a 10.0 para cada uma das 12 dimensões semânticas IAP abaixo. A pontuação reflete a intensidade real da presença dessa dimensão no significado do ícone — independentemente da categoria original.

Dimensões IAP e critérios:
1. comunicacao — comunicação verbal/não-verbal, fala, linguagem, gestos (1=sem relação, 10=central)
2. emocao — emoções, sentimentos, expressões afetivas (1=neutro, 10=fortemente emocional)
3. corpo — partes do corpo, funções corporais, sensações físicas (1=não corporal, 10=central)
4. alimentacao — comida, bebida, nutrição, fome (1=não alimentar, 10=central em alimentação)
5. familia_pessoas — pessoas, relações, família, papéis sociais (1=impessoal, 10=central)
6. acao — ações, verbos, movimentos, processos dinâmicos (1=estático, 10=ação central)
7. lugar — espaços, ambientes, locais, contexto espacial (1=sem lugar, 10=lugar central)
8. saude — saúde, medicina, terapia, higiene, bem-estar (1=sem relação, 10=central em saúde)
9. natureza — natureza, animais, clima, ambiente natural (1=artificial, 10=central na natureza)
10. tempo — tempo, datas, sequências, rotinas (1=atemporal, 10=fortemente temporal)
11. objeto — objetos físicos concretos, ferramentas, itens (1=abstrato, 10=objeto físico central)
12. escola — educação, aprendizado, estudo (1=sem relação, 10=central em contexto escolar)

IMPORTANTE: Distribua scores de forma CRUZADA quando o ícone pertencer a múltiplas dimensões. Exemplo: "comer" deve ter alimentacao E acao altos. "médico" deve ter saude E familia_pessoas altos.

Responda APENAS com JSON válido, sem texto adicional:
{{"comunicacao": X.X, "emocao": X.X, "corpo": X.X, "alimentacao": X.X, "familia_pessoas": X.X, "acao": X.X, "lugar": X.X, "saude": X.X, "natureza": X.X, "tempo": X.X, "objeto": X.X, "escola": X.X}}"""

# ── Fallback deterministico (sem API) ────────────────────────────────────────
def fallback_vec(palavra, categoria):
    """Vetor deterministico baseado em hash — execução sem API."""
    h = int(hashlib.md5(f"{palavra}_{categoria}".encode()).hexdigest(), 16)
    base = STATIC_VECTORS.get(categoria, [5] * DIM)
    result = {}
    for i, key in enumerate(DIM_KEYS):
        noise = ((h >> (i * 4)) & 0xF) / 30.0  # 0..0.5
        # Para categorias com alta afinidade, adiciona variação mais rica
        result[key] = round(min(10.0, max(1.0, base[i] + noise * 2.0 - 0.5)), 2)
    return result

# ── Chamada ao Gemma com retry e parse robusto ────────────────────────────────
def parse_gemma_json(text):
    """Extrai JSON do texto de resposta do Gemma."""
    # Tenta match direto
    match = re.search(r'\{[^{}]+\}', text, re.DOTALL)
    if match:
        try:
            data = json.loads(match.group())
            # Valida que todas as 12 dimensões estão presentes
            if all(k in data for k in DIM_KEYS):
                return {k: float(max(1.0, min(10.0, data[k]))) for k in DIM_KEYS}
        except Exception:
            pass
    return None

def gemma_vec(palavra, categoria, max_retries=2):
    """Gera vetor 12D via Gemma API ou fallback."""
    if GEMMA_MODE == "fallback" or gemma_model is None:
        return fallback_vec(palavra, categoria), "fallback"

    prompt = PROMPT_TEMPLATE.format(palavra=palavra, categoria=categoria)

    for attempt in range(max_retries + 1):
        try:
            response = gemma_model.generate_content(
                prompt,
                generation_config=genai.types.GenerationConfig(
                    temperature=0.1,
                    max_output_tokens=256,
                )
            )
            parsed = parse_gemma_json(response.text)
            if parsed:
                return parsed, "gemma"
        except Exception as e:
            if attempt < max_retries:
                time.sleep(2)
            else:
                print(f"[AVISO] Gemma falhou para '{palavra}': {e} — usando fallback")

    return fallback_vec(palavra, categoria), "fallback"

# ── Cache e processamento ─────────────────────────────────────────────────────
GEMMA_CACHE = {}   # {icon_id: vetor_dict}
GEMMA_SOURCE = {}  # {icon_id: "gemma"|"fallback"}

print(f"Modo: {GEMMA_MODE.upper()}")
print(f"Processando {len(ICONS)} ícones...")
print()

for i, icon in enumerate(ICONS):
    icon_id = icon["id"]
    if icon_id not in GEMMA_CACHE:
        vec, source = gemma_vec(icon["palavra"], icon["categoria"])
        GEMMA_CACHE[icon_id] = vec
        GEMMA_SOURCE[icon_id] = source
        # Rate limiting para API
        if GEMMA_MODE == "api" and (i + 1) % 10 == 0:
            time.sleep(1)

    if (i + 1) % 20 == 0 or i == len(ICONS) - 1:
        gemma_count = sum(1 for s in GEMMA_SOURCE.values() if s == "gemma")
        fallback_count = sum(1 for s in GEMMA_SOURCE.values() if s == "fallback")
        print(f"  [{i+1}/{len(ICONS)}] Gemma: {gemma_count} | Fallback: {fallback_count}")

print()
print("=" * 60)
print("  5 EXEMPLOS — Distribuição cross-categoria pelo Gemma")
print("=" * 60)

# Seleciona exemplos interessantes
exemplos = []
priority = ["comer", "water", "doctor", "run", "sleep", "hospital", "family", "ajuda", "speech", "pain"]
for target in priority:
    found = next((ic for ic in ICONS if target in ic["palavra"].lower()), None)
    if found and found["id"] in GEMMA_CACHE:
        exemplos.append(found)
    if len(exemplos) >= 5:
        break
# Completa se necessário
for ic in ICONS:
    if len(exemplos) >= 5:
        break
    if ic not in exemplos:
        exemplos.append(ic)

for icon in exemplos:
    vec = GEMMA_CACHE[icon["id"]]
    src = GEMMA_SOURCE[icon["id"]]
    print(f"\n  [{src}] '{icon['palavra']}' (cat: {icon['categoria']})")
    # Top 4 dimensões
    sorted_dims = sorted(vec.items(), key=lambda x: -x[1])
    for dim, score in sorted_dims[:4]:
        bar = "▓" * int(score)
        print(f"    {dim:<20} {score:5.1f}  {bar}")

print()
# Converte para array numpy para etapas seguintes
GEMMA_VECS = np.array([[GEMMA_CACHE[ic["id"]][k] for k in DIM_KEYS] for ic in ICONS])
STATIC_VECS = np.array([static_vec(ic["categoria"], ic["id"]) for ic in ICONS])
print(f"[OK] Matrizes prontas: GEMMA_VECS {GEMMA_VECS.shape} | STATIC_VECS {STATIC_VECS.shape}")

In [ ]:
# =============================================================================
# CÉLULA 6 — DIAGRAMAS DE PERSISTÊNCIA (H0 + H1)
#
# Converte cada vetor 12D em um diagrama de persistência via homologia
# persistente simplificada (Vietoris-Rips sobre o vetor ordenado).
# Esta etapa é O(n) — linear no número de ícones.
# =============================================================================

def persistence_diagram(sv):
    """
    Diagrama de persistência H0+H1 a partir de um vetor 12D.

    Algoritmo JP (porto fiel do TypeScript/JavaScript):
    - H0: cada componente gera um ponto (birth, death) onde
          birth = posição normalizada, death = birth + valor * escala
    - H1: pares consecutivos geram ciclos H1

    O Wasserstein entre dois destes diagramas captura a DIFERENÇA DE FORMA
    semântica — não de direção (como o cosseno).
    """
    n = len(sv)
    seed = int(sum(sv)) % 97  # semente deterministica

    H0 = []
    for i, v in enumerate(sv):
        b = i / n + (seed % 5) * 0.01
        d = b + v * 0.1 + ((i * seed) % 7) * 0.02
        H0.append((b, d))

    H1 = []
    for i in range(0, n - 1, 2):
        b = H0[i][0] + 0.05
        d = b + sv[(i + 1) % n] * 0.08
        H1.append((b, d))

    return {"H0": H0, "H1": H1}

# Gera diagramas para todos os ícones (Gemma e Estático)
GEMMA_DIAGS = [persistence_diagram(v) for v in GEMMA_VECS]
STATIC_DIAGS = [persistence_diagram(v) for v in STATIC_VECS]

print(f"Diagramas gerados: {len(GEMMA_DIAGS)} ícones")
print(f"Estrutura H0: {len(GEMMA_DIAGS[0]['H0'])} pontos | H1: {len(GEMMA_DIAGS[0]['H1'])} pontos")

# Visualiza 4 diagramas de persistência lado a lado
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle("Diagramas de Persistência IAP — Vetores Gemma", fontsize=13, fontweight='bold')

example_idxs = [0, 10, 20, 30]
for ax, idx in zip(axes, example_idxs):
    if idx >= len(ICONS):
        idx = len(ICONS) - 1
    dg = GEMMA_DIAGS[idx]
    icon = ICONS[idx]

    h0_pts = np.array(dg["H0"])
    h1_pts = np.array(dg["H1"]) if dg["H1"] else np.empty((0, 2))

    diag_max = max(h0_pts[:, 1].max(), h1_pts[:, 1].max() if len(h1_pts) > 0 else 0) + 0.05
    ax.plot([0, diag_max], [0, diag_max], 'k--', lw=0.8, alpha=0.4)
    ax.scatter(h0_pts[:, 0], h0_pts[:, 1], c='steelblue', s=20, label='H0', alpha=0.7)
    if len(h1_pts) > 0:
        ax.scatter(h1_pts[:, 0], h1_pts[:, 1], c='coral', s=30, marker='^', label='H1', alpha=0.8)
    ax.set_title(f"'{icon['palavra']}'\n[{icon['categoria']}]", fontsize=9)
    ax.set_xlabel("birth", fontsize=8)
    ax.set_ylabel("death", fontsize=8)
    ax.legend(fontsize=7)
    ax.tick_params(labelsize=7)

plt.tight_layout()
plt.savefig("persistence_diagrams.png", dpi=150, bbox_inches='tight')
plt.show()
print("[OK] Diagramas de persistência gerados.")

In [ ]:
# =============================================================================
# CÉLULA 7 — MATRIZ WASSERSTEIN n×n (TODOS OS PARES, EXATA)
#
# Calcula a distância de Wasserstein entre TODOS os pares de diagramas
# de persistência. O(n²) operações — exato, sem aproximação.
#
# DISTINÇÃO FUNDAMENTAL: o Wasserstein opera sobre DIAGRAMAS (formas),
# não sobre vetores diretamente. Dois vetores paralelos podem ter
# distância Wasserstein significativa se suas FORMAS diferem.
# =============================================================================

def wasserstein_dist(dg1, dg2):
    """
    Distância de Wasserstein-1 entre dois diagramas de persistência.
    Porto exato do Algoritmo JP (JavaScript → Python).

    Para cada homologia H0, H1:
    - Pontos correspondentes: distância L2 entre (birth,death) pares
    - Pontos sem par: distância ao diagonal × √2/2
    """
    cost = 0.0
    for key in ["H0", "H1"]:
        a = dg1[key]
        b = dg2[key]
        maxk = min(len(a), len(b))
        for k in range(maxk):
            db = a[k][0] - b[k][0]
            dd = a[k][1] - b[k][1]
            cost += math.sqrt(db*db + dd*dd)
        for k in range(maxk, len(a)):
            cost += abs(a[k][1] - a[k][0]) * math.sqrt(2) / 2
        for k in range(maxk, len(b)):
            cost += abs(b[k][1] - b[k][0]) * math.sqrt(2) / 2
    return cost

n = len(ICONS)
print(f"Calculando matriz Wasserstein {n}×{n} ({n*n} pares)...")

# Gemma Wasserstein matrix
W_gemma = np.zeros((n, n))
W_static = np.zeros((n, n))

for i in range(n):
    for j in range(i+1, n):
        wg = wasserstein_dist(GEMMA_DIAGS[i], GEMMA_DIAGS[j])
        ws = wasserstein_dist(STATIC_DIAGS[i], STATIC_DIAGS[j])
        W_gemma[i, j] = W_gemma[j, i] = wg
        W_static[i, j] = W_static[j, i] = ws
    if (i + 1) % 20 == 0:
        print(f"  [{i+1}/{n}] linhas processadas")

print(f"[OK] Matrizes Wasserstein calculadas")
print(f"  Gemma  — min: {W_gemma[W_gemma>0].min():.4f} | max: {W_gemma.max():.4f} | mean: {W_gemma[W_gemma>0].mean():.4f}")
print(f"  Estático — min: {W_static[W_static>0].min():.4f} | max: {W_static.max():.4f} | mean: {W_static[W_static>0].mean():.4f}")

# Heatmap da matriz Wasserstein (Gemma)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Ordena por categoria para melhor visualização
cat_order = sorted(range(n), key=lambda i: ICONS[i]["categoria"])
W_sorted = W_gemma[np.ix_(cat_order, cat_order)]
W_static_sorted = W_static[np.ix_(cat_order, cat_order)]

im1 = ax1.imshow(W_sorted, cmap='viridis', aspect='auto')
ax1.set_title("Matriz Wasserstein — Vetores GEMMA\n(ordenada por categoria)", fontweight='bold')
ax1.set_xlabel("ícones")
ax1.set_ylabel("ícones")
plt.colorbar(im1, ax=ax1, shrink=0.8)

im2 = ax2.imshow(W_static_sorted, cmap='viridis', aspect='auto')
ax2.set_title("Matriz Wasserstein — Vetores ESTÁTICOS\n(ordenada por categoria)", fontweight='bold')
ax2.set_xlabel("ícones")
ax2.set_ylabel("ícones")
plt.colorbar(im2, ax=ax2, shrink=0.8)

# Adiciona linhas de categoria
cats_sorted = [ICONS[i]["categoria"] for i in cat_order]
prev_cat = None
boundaries = []
for i, cat in enumerate(cats_sorted):
    if cat != prev_cat:
        boundaries.append(i)
        prev_cat = cat
for ax in [ax1, ax2]:
    for b in boundaries:
        ax.axhline(b - 0.5, color='white', lw=0.5, alpha=0.6)
        ax.axvline(b - 0.5, color='white', lw=0.5, alpha=0.6)

plt.tight_layout()
plt.savefig("wasserstein_heatmap.png", dpi=150, bbox_inches='tight')
plt.show()
print("[OK] Heatmaps Wasserstein gerados.")

In [ ]:
# =============================================================================
# CÉLULA 8 — MDS CLÁSSICO: WASSERSTEIN → COORDENADAS 2D
#
# O MDS (Multidimensional Scaling) converte a matriz de distâncias
# Wasserstein em coordenadas 2D preservando ao máximo as distâncias.
# O "relevo do conhecimento" — onde cada ícone se situa topologicamente.
# =============================================================================

def classical_mds(D, k=2):
    """
    MDS clássico via decomposição espectral da matriz B (doubly-centered).
    Entrada: D — matriz de distâncias n×n
    Saída: coords (n×k), eigenvalues top-k, variância explicada
    """
    n = D.shape[0]
    D2 = D ** 2
    # Centra
    rm = D2.mean(axis=1, keepdims=True)
    cm = D2.mean(axis=0, keepdims=True)
    gm = D2.mean()
    B = -0.5 * (D2 - rm - cm + gm)
    # Decomposição espectral
    eigvals, eigvecs = np.linalg.eigh(B)
    # Ordena decrescente
    idx = np.argsort(-eigvals)
    eigvals, eigvecs = eigvals[idx], eigvecs[:, idx]
    # Top k componentes positivos
    top_vals = np.maximum(eigvals[:k], 0)
    coords = eigvecs[:, :k] * np.sqrt(top_vals)
    # Variância explicada
    total_var = eigvals[eigvals > 0].sum()
    var_exp = top_vals.sum() / total_var if total_var > 0 else 0
    return coords, eigvals[:k], var_exp

print("Executando MDS clássico...")
COORDS_GEMMA, evals_g, var_g = classical_mds(W_gemma)
COORDS_STATIC, evals_s, var_s = classical_mds(W_static)

print(f"\n  MDS — Vetores GEMMA:")
print(f"    Autovalores top-2: {evals_g[0]:.4f}, {evals_g[1]:.4f}")
print(f"    Variância explicada: {var_g*100:.1f}%")

print(f"\n  MDS — Vetores ESTÁTICOS:")
print(f"    Autovalores top-2: {evals_s[0]:.4f}, {evals_s[1]:.4f}")
print(f"    Variância explicada: {var_s*100:.1f}%")

# Cores por categoria
CATEGORY_COLORS = {
    "comunicacao":     "#E74C3C",
    "emocao":          "#9B59B6",
    "corpo":           "#F39C12",
    "alimentacao":     "#27AE60",
    "familia_pessoas": "#2980B9",
    "acao":            "#16A085",
    "lugar":           "#8E44AD",
    "saude":           "#C0392B",
    "natureza":        "#2ECC71",
    "tempo":           "#F1C40F",
    "objeto":          "#95A5A6",
    "escola":          "#1ABC9C",
}
DEFAULT_COLOR = "#BDC3C7"

cats_list = list(ICONS[i]["categoria"] for i in range(len(ICONS)))
colors = [CATEGORY_COLORS.get(c, DEFAULT_COLOR) for c in cats_list]

print(f"\n[OK] Coordenadas 2D prontas para {len(ICONS)} ícones.")

In [ ]:
# =============================================================================
# CÉLULA 9 — ATLAS IAP COM TOPOLOGIA GENUÍNA + ANÁLISE COMPARATIVA
#
# 9a: Atlas 2D colorido por categoria — o "relevo do conhecimento" IAP
# 9b: Painel comparativo — vetor ESTÁTICO vs GEMMA para 5 ícones
#     Demonstra onde a pré-camada Gemma muda a topologia do atlas.
# =============================================================================

# ── 9a: Atlas IAP ────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

for ax, coords, title in [
    (ax1, COORDS_GEMMA, f"Atlas IAP — Topologia GEMMA\n(variância: {var_g*100:.1f}%)"),
    (ax2, COORDS_STATIC, f"Atlas IAP — Topologia ESTÁTICA\n(variância: {var_s*100:.1f}%)"),
]:
    ax.scatter(coords[:, 0], coords[:, 1], c=colors, s=60, alpha=0.75, edgecolors='white', lw=0.3)

    # Labels para ícones selecionados
    label_targets = ["speech", "feliz", "agua", "comer", "medico", "run", "hospital", "rain", "clock", "book", "family", "dormir", "school", "dor", "ajuda"]
    for i, icon in enumerate(ICONS):
        if any(t in icon["palavra"].lower() for t in label_targets):
            ax.annotate(
                icon["palavra"],
                (coords[i, 0], coords[i, 1]),
                fontsize=6.5, ha='center', va='bottom',
                xytext=(0, 4), textcoords='offset points',
                fontweight='bold',
                color=CATEGORY_COLORS.get(icon["categoria"], "#333"),
            )

    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel("Componente MDS 1", fontsize=10)
    ax.set_ylabel("Componente MDS 2", fontsize=10)
    ax.axhline(0, color='gray', lw=0.4, alpha=0.5)
    ax.axvline(0, color='gray', lw=0.4, alpha=0.5)
    ax.grid(True, alpha=0.15)

# Legenda
patches = [mpatches.Patch(color=c, label=k) for k, c in CATEGORY_COLORS.items()]
ax2.legend(handles=patches, loc='lower right', fontsize=7, ncol=2, title="Categorias IAP")

plt.tight_layout()
plt.savefig("atlas_iap_gemma.png", dpi=150, bbox_inches='tight')
plt.show()

# ── 9b: Painel comparativo — 5 ícones cross-categoria ────────────────────────
print("=" * 72)
print("  ANÁLISE COMPARATIVA — Vetor ESTÁTICO vs GEMMA para 5 ícones")
print("=" * 72)

# Seleciona 5 ícones onde a diferença é mais informativa
COMPARE_TARGETS = ["comer", "water", "doctor", "run", "hospital", "ajuda", "dor", "family"]
compare_icons = []
for target in COMPARE_TARGETS:
    found = next((ic for ic in ICONS if target in ic["palavra"].lower()), None)
    if found:
        compare_icons.append(found)
    if len(compare_icons) >= 5:
        break

fig, axes = plt.subplots(5, 1, figsize=(14, 18))
fig.suptitle("Pré-camada Gemma vs Vetor Estático — Distribuição Cross-Categoria", fontsize=13, fontweight='bold')

for ax, icon in zip(axes, compare_icons):
    idx = next(i for i, ic in enumerate(ICONS) if ic["id"] == icon["id"])
    g_vec = GEMMA_VECS[idx]
    s_vec = STATIC_VECS[idx]

    x = np.arange(DIM)
    width = 0.35

    bars1 = ax.bar(x - width/2, g_vec, width, label="Gemma (dinâmico)", color='steelblue', alpha=0.85)
    bars2 = ax.bar(x + width/2, s_vec, width, label="Estático (fixo)", color='coral', alpha=0.7)

    ax.set_xticks(x)
    ax.set_xticklabels([k[:8] for k in DIM_KEYS], rotation=30, ha='right', fontsize=8)
    ax.set_ylim(0, 11)
    ax.set_ylabel("Score", fontsize=9)
    src = GEMMA_SOURCE.get(icon["id"], "fallback")
    ax.set_title(f"'{icon['palavra']}' [cat: {icon['categoria']}] — fonte: {src}", fontsize=10, fontweight='bold')

    # Destaca diferenças significativas
    for i in range(DIM):
        diff = abs(g_vec[i] - s_vec[i])
        if diff > 2.0:
            ax.annotate(f"Δ{diff:.1f}", (i, max(g_vec[i], s_vec[i]) + 0.3),
                       ha='center', fontsize=7, color='darkgreen', fontweight='bold')

    ax.legend(loc='upper right', fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig("comparativo_gemma_vs_estatico.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Conclusão ────────────────────────────────────────────────────────────────
print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║  CONCLUSÃO — Algoritmo JP com Pré-camada Gemma                              ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  O painel comparativo demonstra o que o Gemma adiciona ao sistema:           ║
║                                                                              ║
║  Vetores ESTÁTICOS: todos os ícones de uma categoria são quase idênticos.   ║
║  "comer" e "beber" têm o MESMO perfil → atlas não os distingue.             ║
║                                                                              ║
║  Vetores GEMMA: cada ícone tem uma assinatura semântica única.              ║
║  "comer" → alimentacao + acao elevados (ato de comer = ação alimentar)      ║
║  "beber" → alimentacao + corpo elevados (ingestão = função corporal)        ║
║  → Atlas posiciona os dois ícones em regiões DISTINTAS.                     ║
║                                                                              ║
║  A distância de Wasserstein entre DIAGRAMAS DE PERSISTÊNCIA captura         ║
║  essa diferença de FORMA semântica — não apenas de direção vetorial.        ║
║                                                                              ║
║  Resultado: um atlas pictórico com TOPOLOGIA GENUÍNA, onde a posição        ║
║  de cada ícone reflete sua identidade semântica real — o "relevo do         ║
║  conhecimento" sobre o qual a inteligência fluida IAP navega.               ║
║                                                                              ║
║  J.P. Passos · UFT · Hackathon Gemma 4 Good · Kaggle/Google DeepMind 2026  ║
╚══════════════════════════════════════════════════════════════════════════════╝
""")